# Objetivo
Refatorar os extractores e transformadores de robocalls para aplicação universal. 

Neste notebook serão testadas, sobre uma amostra dos dados de CDRs, funções para:

- extrair CDRs processados do formato texto para parquet;
- transformar CDRs para formato normalizado (tabela única com campos uniformes);

> Atenção: este notebook deve ser executado no kernel com Python 3.9

# Configuração do ambiente

## Bibliotecas

In [1]:
import logging

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from teleutils import robocalls
from teleutils.core.extractors import CDRTextExtractor
from teleutils.core.transformers import CDRTransformer

## Logging

In [2]:
# Configuração mínima para exibir logs no output da célula
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    datefmt="%H:%M:%S",
)

# Opcional: reduzir ruído de bibliotecas externas (pyspark, py4j, etc.)
logging.getLogger("py4j").setLevel(logging.WARNING)
logging.getLogger("pyspark").setLevel(logging.WARNING)

## Spark Session

### Local

In [3]:
spark = SparkSession.builder \
    .master("local[1]") \
    .appName("testes_chamadas_abusivas") \
    .config("spark.executor.memory", "512m") \
    .getOrCreate()
spark

26/06/11 07:26:38 WARN Utils: Your hostname, LANGTANG resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/06/11 07:26:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/11 07:26:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Pastas origem e destino

In [5]:
# Pasta base origem
INPUT_FOLDER = "/data/cdr/chamadas_abusivas/cdr_processado/Semana86"

# Pasta de arquivos extraídos
EXTRACTED_FOLDER = "/data/cdr/chamadas_abusivas/cdr_extraido/Semana86"

# Pasta de arquivos transformados
TRANSFORMED_FOLDER = "/data/cdr/chamadas_abusivas/cdr_transformado/Semana86"

## Caminho completo dos arquivos

### Pastas locais

In [6]:
INPUT_FOLDER = "/mnt/e/data/datasets/amostra_cdr/semana95"
EXTRACTED_FOLDER = "/home/maxwelfreitas/datasets/cdr_extraido"
TRANSFORMED_FOLDER = "/home/maxwelfreitas/datasets/cdr_transformado"

### Caminho os arquivos

In [29]:
# Claro/Ericsson
cdr_processado_claro_ericsson = f"{INPUT_FOLDER}/claro/ericsson"
cdr_extraido_claro_ericsson = f"{EXTRACTED_FOLDER}/claro_ericsson_extraido.parquet"
cdr_transformado_claro_ericsson = f"{TRANSFORMED_FOLDER}/claro_ericsson_transformado.parquet"

# Claro/Nokia
cdr_processado_claro_nokia = f"{INPUT_FOLDER}/claro/nokia"
cdr_extraido_claro_nokia = f"{EXTRACTED_FOLDER}/claro_nokia_extraido.parquet"
cdr_transformado_claro_nokia = f"{TRANSFORMED_FOLDER}/claro_nokia_transformado.parquet"

# Tim/Ericsson
cdr_processado_tim_ericsson = f"{INPUT_FOLDER}/tim/ericsson"
cdr_extraido_tim_ericsson = f"{EXTRACTED_FOLDER}/tim_ericsson_extraido.parquet"
cdr_transformado_tim_ericsson = f"{TRANSFORMED_FOLDER}/tim_ericsson_transformado.parquet"
cdr_ofensores_tim_ericsson = f"{TRANSFORMED_FOLDER}/tim_ericsson_ofensores.parquet"

# Vivo/Ericsson
cdr_processado_vivo_ericsson = f"{INPUT_FOLDER}/vivo/ericsson"
cdr_extraido_vivo_ericsson = f"{EXTRACTED_FOLDER}/vivo_ericsson_extraido.parquet"
cdr_transformado_vivo_ericsson = f"{TRANSFORMED_FOLDER}/vivo_ericsson_transformado.parquet"
cdr_ofensores_vivo_ericsson = f"{TRANSFORMED_FOLDER}/vivo_ericsson_ofensores.parquet"

# Tim/ATS
cdr_processado_tim_ats = f"{INPUT_FOLDER}/tim/ats"
cdr_extraido_tim_ats = f"{EXTRACTED_FOLDER}/tim_ats_extraido.parquet"
cdr_transformado_tim_ats = f"{TRANSFORMED_FOLDER}/tim_ats_transformado.parquet"
cdr_ofensores_tim_ats = f"{TRANSFORMED_FOLDER}/tim_ats_ofensores.parquet"

# Vivo/FCDR
cdr_processado_vivo_fcdr = f"{INPUT_FOLDER}/vivo/fcdr"
cdr_extraido_vivo_fcdr = f"{EXTRACTED_FOLDER}/vivo_fcdr_extraido.parquet"
cdr_transformado_vivo_fcdr = f"{TRANSFORMED_FOLDER}/vivo_fcdr_transformado.parquet"
cdr_ofensores_vivo_fcdr = f"{TRANSFORMED_FOLDER}/vivo_fcdr_ofensores.parquet"

# Testes

## Extração

In [8]:
extractor = CDRTextExtractor(spark)
extractor

### Ericsson

In [9]:
df_extraido_claro_ericsson = extractor.extract_cdr_ericsson(cdr_processado_claro_ericsson, cdr_extraido_claro_ericsson)
df_extraido_claro_ericsson.show(5)

07:34:38 | INFO | teleutils.core.extractors._text | Iniciando operação [extract_cdr_ericsson]: /mnt/e/data/datasets/amostra_cdr/semana95/claro/ericsson
07:34:38 | INFO | teleutils.core.extractors._text | Lendo arquivo CSV: /mnt/e/data/datasets/amostra_cdr/semana95/claro/ericsson com delimitador ';' e header=True
07:34:44 | INFO | teleutils.core.extractors._text | Validando índices de coluna para o esquema 'Ericsson'
07:34:44 | INFO | teleutils.core.extractors._text | Selecionando e renomeando colunas conforme o esquema 'Ericsson'
07:34:44 | INFO | teleutils.core.extractors._text | Escrevendo DataFrame extraído para parquet particionado por 'tipo_chamada': /home/maxwelfreitas/datasets/cdr_extraido/claro_ericsson_extraido.parquet
07:34:49 | INFO | teleutils.core.extractors._text | Operação [extract_cdr_ericsson] concluída com sucesso.


+----------+-------------+----------+--------+----------------+-------+----------+--------+--------------------+------------+
|referencia|numero_origem|     _data|   _hora|  numero_destino|duracao|prestadora|tipo_cdr|      arquivo_origem|tipo_chamada|
+----------+-------------+----------+--------+----------------+-------+----------+--------+--------------------+------------+
|0000da0004| 27998976734f|2026-03-25|17:16:28|  02121972501589|     76|     claro|ericsson|file:///mnt/e/dat...|         TRA|
|0002c60004| 27997555966f|2026-03-25|17:17:03|    051991118732|      0|     claro|ericsson|file:///mnt/e/dat...|         TRA|
|0003150007| 27997504402f|2026-03-25|16:46:35|   5511990101290|     41|     claro|ericsson|file:///mnt/e/dat...|         TRA|
|0003150007| 27997504402f|2026-03-25|16:46:35|    011990101290|     41|     claro|ericsson|file:///mnt/e/dat...|         TRA|
|0003150007| 27997504402f|2026-03-25|16:46:35|c145511990101290|     41|     claro|ericsson|file:///mnt/e/dat...|      

In [10]:
df_extraido_tim_ericsson = extractor.extract_cdr_ericsson(cdr_processado_tim_ericsson, cdr_extraido_tim_ericsson)
df_extraido_tim_ericsson.show(5)

07:35:03 | INFO | teleutils.core.extractors._text | Iniciando operação [extract_cdr_ericsson]: /mnt/e/data/datasets/amostra_cdr/semana95/tim/ericsson
07:35:03 | INFO | teleutils.core.extractors._text | Lendo arquivo CSV: /mnt/e/data/datasets/amostra_cdr/semana95/tim/ericsson com delimitador ';' e header=True
07:35:05 | INFO | teleutils.core.extractors._text | Validando índices de coluna para o esquema 'Ericsson'
07:35:05 | INFO | teleutils.core.extractors._text | Selecionando e renomeando colunas conforme o esquema 'Ericsson'
07:35:05 | INFO | teleutils.core.extractors._text | Escrevendo DataFrame extraído para parquet particionado por 'tipo_chamada': /home/maxwelfreitas/datasets/cdr_extraido/tim_ericsson_extraido.parquet
07:35:16 | INFO | teleutils.core.extractors._text | Operação [extract_cdr_ericsson] concluída com sucesso.


+----------+-------------+----------+--------+--------------------+-------+----------+--------+--------------------+------------+
|referencia|numero_origem|     _data|   _hora|      numero_destino|duracao|prestadora|tipo_cdr|      arquivo_origem|tipo_chamada|
+----------+-------------+----------+--------+--------------------+-------+----------+--------+--------------------+------------+
|0001fe063e| 27981236722f|2026-03-25|15:09:57|           988671912|      0|       tim|ericsson|file:///mnt/e/dat...|         TRA|
|00025b3149| 27981080037f|2026-03-25|15:56:28|      04127981383871|      0|       tim|ericsson|file:///mnt/e/dat...|         TRA|
|00025b3149| 27981080037f|2026-03-25|15:56:28|041a0341000002798...|      0|       tim|ericsson|file:///mnt/e/dat...|         TRA|
|00025b3149| 27981080037f|2026-03-25|15:56:28|       ae27981383871|      0|       tim|ericsson|file:///mnt/e/dat...|         TRA|
|00025b3149| 27981080037f|2026-03-25|15:56:28|041a0341000002798...|      0|       tim|eric

In [11]:
df_extraido_vivo_ericsson = extractor.extract_cdr_ericsson(cdr_processado_vivo_ericsson, cdr_extraido_vivo_ericsson)
df_extraido_vivo_ericsson.show(5)

07:35:16 | INFO | teleutils.core.extractors._text | Iniciando operação [extract_cdr_ericsson]: /mnt/e/data/datasets/amostra_cdr/semana95/vivo/ericsson
07:35:16 | INFO | teleutils.core.extractors._text | Lendo arquivo CSV: /mnt/e/data/datasets/amostra_cdr/semana95/vivo/ericsson com delimitador ';' e header=True
07:35:17 | INFO | teleutils.core.extractors._text | Validando índices de coluna para o esquema 'Ericsson'
07:35:17 | INFO | teleutils.core.extractors._text | Selecionando e renomeando colunas conforme o esquema 'Ericsson'
07:35:18 | INFO | teleutils.core.extractors._text | Escrevendo DataFrame extraído para parquet particionado por 'tipo_chamada': /home/maxwelfreitas/datasets/cdr_extraido/vivo_ericsson_extraido.parquet
07:35:27 | INFO | teleutils.core.extractors._text | Operação [extract_cdr_ericsson] concluída com sucesso.


+----------+-------------+----------+--------+--------------+-------+----------+--------+--------------------+------------+
|referencia|numero_origem|     _data|   _hora|numero_destino|duracao|prestadora|tipo_cdr|      arquivo_origem|tipo_chamada|
+----------+-------------+----------+--------+--------------+-------+----------+--------+--------------------+------------+
|00024b0003| 27996481602f|2026-03-25|14:10:43|  066999060988|      1|      vivo|ericsson|file:///mnt/e/dat...|         TRA|
|0002860006| 27998128912f|2026-03-25|15:11:04|  022996240373|      0|      vivo|ericsson|file:///mnt/e/dat...|         TRA|
|0003d10006| 27997535377f|2026-03-25|15:11:42|  021972437287|      0|      vivo|ericsson|file:///mnt/e/dat...|         TRA|
|0006680006| 27996860679f|2026-03-25|15:12:36|  022996240168|      0|      vivo|ericsson|file:///mnt/e/dat...|         TRA|
|0006e60006| 27997882173f|2026-03-25|15:12:24|  022996240300|      0|      vivo|ericsson|file:///mnt/e/dat...|         TRA|
+-------

### Tim/ATS

In [12]:
df_extraido_tim_ats = extractor.extract_cdr_tim_ats(cdr_processado_tim_ats,cdr_extraido_tim_ats)
df_extraido_tim_ats.show(5)

07:35:32 | INFO | teleutils.core.extractors._text | Iniciando operação [extract_cdr_tim_ats]: /mnt/e/data/datasets/amostra_cdr/semana95/tim/ats
07:35:33 | INFO | teleutils.core.extractors._text | Lendo arquivo CSV: /mnt/e/data/datasets/amostra_cdr/semana95/tim/ats com delimitador ';' e header=False
07:35:33 | INFO | teleutils.core.extractors._text | Validando índices de coluna para o esquema 'Tim ATS'
07:35:33 | INFO | teleutils.core.extractors._text | Selecionando e renomeando colunas conforme o esquema 'Tim ATS'
07:35:33 | INFO | teleutils.core.extractors._text | Aplicando filtro: tipo_chamada = 'TipodeCDR(role-of-Node)' para o esquema 'Tim ATS'
07:35:33 | INFO | teleutils.core.extractors._text | Escrevendo DataFrame extraído para parquet particionado por 'tipo_chamada': /home/maxwelfreitas/datasets/cdr_extraido/tim_ats_extraido.parquet
07:35:35 | INFO | teleutils.core.extractors._text | Operação [extract_cdr_tim_ats] concluída com sucesso.


+--------------+----------+--------+--------------+-------+----------+--------------------+----------+--------+--------------------+------------+
| numero_origem|     _data|   _hora|numero_destino|duracao|referencia|       _autenticacao|prestadora|tipo_cdr|      arquivo_origem|tipo_chamada|
+--------------+----------+--------+--------------+-------+----------+--------------------+----------+--------+--------------------+------------+
|5527981251935F|2026-03-25|15-35-37|5527992760708F|   NULL|04918DF370|verstat=TN-Valida...|       tim|     ats|file:///mnt/e/dat...|        TERv|
|5527981175358F|2026-03-25|15-35-37|5527996580276F|   NULL|04A18E08DB|verstat=No-TN-Val...|       tim|     ats|file:///mnt/e/dat...|        TERv|
|5527981252483F|2026-03-25|15-35-39|5527999226966F|   NULL|02B18DF277|                NULL|       tim|     ats|file:///mnt/e/dat...|        TERv|
|5527981236676F|2026-03-25|15-35-40|5527992296183F|   NULL|00E18E1020|verstat=TN-Valida...|       tim|     ats|file:///mnt/e

### Vivo/FCDR

In [15]:
df_extraido_vivo_fcdr = extractor.extract_cdr_vivo_fcdr(cdr_processado_vivo_fcdr,cdr_extraido_vivo_fcdr)
df_extraido_vivo_fcdr.show(5)

07:36:19 | INFO | teleutils.core.extractors._text | Iniciando operação [extract_cdr_vivo_fcdr]: /mnt/e/data/datasets/amostra_cdr/semana95/vivo/fcdr
07:36:19 | INFO | teleutils.core.extractors._text | Lendo arquivo CSV: /mnt/e/data/datasets/amostra_cdr/semana95/vivo/fcdr com delimitador '|' e header=False
07:36:20 | INFO | teleutils.core.extractors._text | Validando índices de coluna para o esquema 'Vivo FCDR'
07:36:20 | INFO | teleutils.core.extractors._text | Selecionando e renomeando colunas conforme o esquema 'Vivo FCDR'
07:36:20 | INFO | teleutils.core.extractors._text | Escrevendo DataFrame extraído para parquet particionado por 'tipo_chamada': /home/maxwelfreitas/datasets/cdr_extraido/vivo_fcdr_extraido.parquet
07:36:30 | INFO | teleutils.core.extractors._text | Operação [extract_cdr_vivo_fcdr] concluída com sucesso.


+--------------------+--------------+-------+--------+------+------------+----------+--------+--------------------+------------+
|      _numero_origem|numero_destino|duracao|   _data| _hora|  referencia|prestadora|tipo_cdr|      arquivo_origem|tipo_chamada|
+--------------------+--------------+-------+--------+------+------------+----------+--------+--------------------+------------+
|5566996620871;ver...| 5566999845034|      7|20260325|151253|33E39BE413B2|      vivo|    fcdr|file:///mnt/e/dat...|           4|
|       5566984331196| 5566999529195|      0|20260325|151301|24ADD4F70E47|      vivo|    fcdr|file:///mnt/e/dat...|           4|
|5566999353503;ver...| 5566996132543|      0|20260325|151301|15D403700869|      vivo|    fcdr|file:///mnt/e/dat...|           4|
|       5566981403777| 5566999089814|      0|20260325|151301|4D8641BB1E08|      vivo|    fcdr|file:///mnt/e/dat...|           4|
|5566976031704;ver...| 5566996715456|      0|20260325|151301|60D0446A2177|      vivo|    fcdr|fil

### Claro/Nokia

In [30]:
df_extraido_claro_nokia = extractor.extract_cdr_claro_nokia(cdr_processado_claro_nokia,cdr_extraido_claro_nokia)
df_extraido_claro_nokia.show(5)

08:06:23 | INFO | teleutils.core.extractors._text | Iniciando operação [extract_cdr_claro_nokia]: /mnt/e/data/datasets/amostra_cdr/semana95/claro/nokia
08:06:23 | INFO | teleutils.core.extractors._text | Lendo arquivo CSV: /mnt/e/data/datasets/amostra_cdr/semana95/claro/nokia com delimitador ';' e header=True
08:06:25 | INFO | teleutils.core.extractors._text | Validando índices de coluna para o esquema 'Claro Nokia'
08:06:25 | INFO | teleutils.core.extractors._text | Selecionando e renomeando colunas conforme o esquema 'Claro Nokia'
08:06:25 | INFO | teleutils.core.extractors._text | Escrevendo DataFrame extraído para parquet particionado por 'tipo_chamada': /home/maxwelfreitas/datasets/cdr_extraido/claro_nokia_extraido.parquet
08:06:42 | INFO | teleutils.core.extractors._text | Operação [extract_cdr_claro_nokia] concluída com sucesso.


+----------+-------------------+-------------+--------------+-------+----------+--------+--------------------+------------+
|referencia|          data_hora|numero_origem|numero_destino|duracao|prestadora|tipo_cdr|      arquivo_origem|tipo_chamada|
+----------+-------------------+-------------+--------------+-------+----------+--------+--------------------+------------+
|D5243C6145|2026-03-25 17:49:37|   2721216951|   27988775283|     37|     claro|   nokia|file:///mnt/e/dat...|         POC|
|01079A5007|2026-03-25 16:43:10|  27996481602|04161991844980|     26|     claro|   nokia|file:///mnt/e/dat...|         POC|
|D5243C6182|2026-03-25 14:58:38|   2721219422|   27992676717|     10|     claro|   nokia|file:///mnt/e/dat...|         POC|
|11070A60A3|2026-03-25 15:35:19|  27999144454| 5561991722458|    178|     claro|   nokia|file:///mnt/e/dat...|         POC|
|D5243C7003|2026-03-25 15:48:17|   2721210987|   27981467157|     37|     claro|   nokia|file:///mnt/e/dat...|         POC|
+-------

## Transformação

In [17]:
transformer = CDRTransformer(spark)
transformer

### Ericsson

In [32]:
df = transformer.transform_cdr_ericsson(cdr_extraido_claro_ericsson, cdr_transformado_claro_ericsson)
df.show(5)
df.printSchema()

08:09:04 | INFO | teleutils.core.transformers | Iniciando operação [transform_cdr_ericsson]: /home/maxwelfreitas/datasets/cdr_extraido/claro_ericsson_extraido.parquet
08:09:07 | INFO | teleutils.core.transformers | Operação [transform_cdr_ericsson] concluída com sucesso.


+-------------+------------------+-------------------+-----------+----------------+-----------+-----------------+-------------------+-------------------+---------------+---------------+-------------+-----------+--------------------+
|nu_referencia|nu_origem_original|nu_destino_original|  nu_origem|ic_origem_valido| nu_destino|ic_destino_valido|         dh_chamada|qt_duracao_segundos|no_tipo_chamada|no_autenticacao|no_prestadora|no_tipo_cdr|   no_arquivo_origem|
+-------------+------------------+-------------------+-----------+----------------+-----------+-----------------+-------------------+-------------------+---------------+---------------+-------------+-----------+--------------------+
|   0000da0004|      27998976734f|     02121972501589|27998976734|            true|21972501589|             true|2026-03-25 17:16:28|                 76|            TRA|           NULL|        claro|   ericsson|file:///mnt/e/dat...|
|   0002c60004|      27997555966f|       051991118732|27997555966|  

In [34]:
df.groupBy("no_tipo_chamada").count().show()

+---------------+-----+
|no_tipo_chamada|count|
+---------------+-----+
|            ORI| 8890|
|            ROA|19273|
|            TER|15998|
|            FOR| 4845|
|            TRA|54713|
+---------------+-----+



In [19]:
df = transformer.transform_cdr_ericsson(cdr_extraido_tim_ericsson,cdr_transformado_tim_ericsson)
df.show(5)
df.printSchema()

07:37:04 | INFO | teleutils.core.transformers | Iniciando operação [transform_cdr_ericsson]: /home/maxwelfreitas/datasets/cdr_extraido/tim_ericsson_extraido.parquet
07:37:18 | INFO | teleutils.core.transformers | Operação [transform_cdr_ericsson] concluída com sucesso.


+-------------+------------------+--------------------+-----------+----------------+-----------+-----------------+-------------------+-------------------+---------------+---------------+-------------+-----------+--------------------+
|nu_referencia|nu_origem_original| nu_destino_original|  nu_origem|ic_origem_valido| nu_destino|ic_destino_valido|         dh_chamada|qt_duracao_segundos|no_tipo_chamada|no_autenticacao|no_prestadora|no_tipo_cdr|   no_arquivo_origem|
+-------------+------------------+--------------------+-----------+----------------+-----------+-----------------+-------------------+-------------------+---------------+---------------+-------------+-----------+--------------------+
|   0001fe063e|      27981236722f|           988671912|27981236722|            true|  988671912|             true|2026-03-25 15:09:57|                  0|            TRA|           NULL|          tim|   ericsson|file:///mnt/e/dat...|
|   00025b3149|      27981080037f|      04127981383871|279810800

In [20]:
df = transformer.transform_cdr_ericsson(cdr_extraido_vivo_ericsson,cdr_transformado_vivo_ericsson)
df.show(5)
df.printSchema()

07:37:18 | INFO | teleutils.core.transformers | Iniciando operação [transform_cdr_ericsson]: /home/maxwelfreitas/datasets/cdr_extraido/vivo_ericsson_extraido.parquet
07:37:29 | INFO | teleutils.core.transformers | Operação [transform_cdr_ericsson] concluída com sucesso.


+-------------+------------------+-------------------+-----------+----------------+-----------+-----------------+-------------------+-------------------+---------------+---------------+-------------+-----------+--------------------+
|nu_referencia|nu_origem_original|nu_destino_original|  nu_origem|ic_origem_valido| nu_destino|ic_destino_valido|         dh_chamada|qt_duracao_segundos|no_tipo_chamada|no_autenticacao|no_prestadora|no_tipo_cdr|   no_arquivo_origem|
+-------------+------------------+-------------------+-----------+----------------+-----------+-----------------+-------------------+-------------------+---------------+---------------+-------------+-----------+--------------------+
|   00024b0003|      27996481602f|       066999060988|27996481602|            true|66999060988|             true|2026-03-25 14:10:43|                  1|            TRA|           NULL|         vivo|   ericsson|file:///mnt/e/dat...|
|   0002860006|      27998128912f|       022996240373|27998128912|  

### Claro/Nokia

In [31]:
df = transformer.transform_cdr_claro_nokia(cdr_extraido_claro_nokia, cdr_transformado_claro_nokia)
df.show(5)
df.printSchema()

08:07:36 | INFO | teleutils.core.transformers | Iniciando operação [transform_cdr_claro_nokia]: /home/maxwelfreitas/datasets/cdr_extraido/claro_nokia_extraido.parquet
08:07:56 | INFO | teleutils.core.transformers | Operação [transform_cdr_claro_nokia] concluída com sucesso.


+-------------+------------------+-------------------+-----------+----------------+-----------+-----------------+-------------------+-------------------+---------------+---------------+-------------+-----------+--------------------+
|nu_referencia|nu_origem_original|nu_destino_original|  nu_origem|ic_origem_valido| nu_destino|ic_destino_valido|         dh_chamada|qt_duracao_segundos|no_tipo_chamada|no_autenticacao|no_prestadora|no_tipo_cdr|   no_arquivo_origem|
+-------------+------------------+-------------------+-----------+----------------+-----------+-----------------+-------------------+-------------------+---------------+---------------+-------------+-----------+--------------------+
|   D5243C6145|        2721216951|        27988775283| 2721216951|            true|27988775283|             true|2026-03-25 17:49:37|                 37|            POC|           NULL|        claro|      nokia|file:///mnt/e/dat...|
|   01079A5007|       27996481602|     04161991844980|27996481602|  

### Tim/ATS

In [21]:
df = transformer.transform_cdr_tim_ats(cdr_extraido_tim_ats,cdr_transformado_tim_ats)
df.show(5)
df.printSchema()

07:37:51 | INFO | teleutils.core.transformers | Iniciando operação [transform_cdr_tim_ats]: /home/maxwelfreitas/datasets/cdr_extraido/tim_ats_extraido.parquet
07:37:52 | INFO | teleutils.core.transformers | Operação [transform_cdr_tim_ats] concluída com sucesso.


+-------------+------------------+-------------------+-----------+----------------+-----------+-----------------+-------------------+-------------------+---------------+--------------------+-------------+-----------+--------------------+
|nu_referencia|nu_origem_original|nu_destino_original|  nu_origem|ic_origem_valido| nu_destino|ic_destino_valido|         dh_chamada|qt_duracao_segundos|no_tipo_chamada|     no_autenticacao|no_prestadora|no_tipo_cdr|   no_arquivo_origem|
+-------------+------------------+-------------------+-----------+----------------+-----------+-----------------+-------------------+-------------------+---------------+--------------------+-------------+-----------+--------------------+
|   04918DF370|    5527981251935F|     5527992760708F|27981251935|            true|27992760708|             true|2026-03-25 15:35:37|                  0|           TERv|TN-Validation-Passed|          tim|        ats|file:///mnt/e/dat...|
|   04A18E08DB|    5527981175358F|     552799658

In [23]:
df.groupBy("no_autenticacao").agg(F.count("nu_referencia")).show()

+--------------------+--------------------+
|     no_autenticacao|count(nu_referencia)|
+--------------------+--------------------+
|                NULL|               53035|
|    No-TN-Validation|               18147|
|TN-Validation-Passed|               21747|
+--------------------+--------------------+



In [27]:
df.groupBy("no_autenticacao").pivot("no_tipo_chamada").agg(F.count("nu_referencia")).show()

+--------------------+-----+----+-----+
|     no_autenticacao| FORv|ORIv| TERv|
+--------------------+-----+----+-----+
|                NULL|20570|7190|25275|
|    No-TN-Validation| 2968|NULL|15179|
|TN-Validation-Passed| 6489|NULL|15258|
+--------------------+-----+----+-----+



### Vivo/FCDR

In [24]:
df_transformado_vivo_fcdr = transformer.transform_cdr_vivo_fcdr(cdr_extraido_vivo_fcdr,cdr_transformado_vivo_fcdr)
df_transformado_vivo_fcdr.show(5)
df_transformado_vivo_fcdr.printSchema()

07:38:39 | INFO | teleutils.core.transformers | Iniciando operação [transform_cdr_vivo_fcdr]: /home/maxwelfreitas/datasets/cdr_extraido/vivo_fcdr_extraido.parquet
07:38:47 | INFO | teleutils.core.transformers | Operação [transform_cdr_vivo_fcdr] concluída com sucesso.


+-------------+------------------+-------------------+-----------+----------------+-----------+-----------------+-------------------+-------------------+---------------+--------------------+-------------+-----------+--------------------+
|nu_referencia|nu_origem_original|nu_destino_original|  nu_origem|ic_origem_valido| nu_destino|ic_destino_valido|         dh_chamada|qt_duracao_segundos|no_tipo_chamada|     no_autenticacao|no_prestadora|no_tipo_cdr|   no_arquivo_origem|
+-------------+------------------+-------------------+-----------+----------------+-----------+-----------------+-------------------+-------------------+---------------+--------------------+-------------+-----------+--------------------+
| 33E39BE413B2|     5566996620871|      5566999845034|66996620871|            true|66999845034|             true|2026-03-25 15:12:53|                  7|              4|TN-Validation-Passed|         vivo|       fcdr|file:///mnt/e/dat...|
| 24ADD4F70E47|     5566984331196|      55669995

In [25]:
df_transformado_vivo_fcdr.groupBy("no_autenticacao").agg(F.count("nu_referencia")).show(truncate=False)

+--------------------+--------------------+
|no_autenticacao     |count(nu_referencia)|
+--------------------+--------------------+
|NULL                |632044              |
|No-TN-Validation    |45104               |
|TN-Validation-Passed|109392              |
|TN-Validation-Failed|31840               |
+--------------------+--------------------+



In [26]:
df_transformado_vivo_fcdr.groupBy("no_autenticacao").pivot("no_tipo_chamada").agg(F.count("nu_referencia")).show(truncate=False)

+--------------------+-----+------+------+
|no_autenticacao     |1    |3     |4     |
+--------------------+-----+------+------+
|NULL                |89724|232187|310133|
|No-TN-Validation    |59   |NULL  |45045 |
|TN-Validation-Passed|169  |NULL  |109223|
|TN-Validation-Failed|55   |NULL  |31785 |
+--------------------+-----+------+------+

